In [ ]:
#doing all the required imports
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

loading the datasets

In [214]:
claims = pd.read_csv("../data/raw/claims.csv")
policies = pd.read_csv("../data/raw/policies.csv")
garages = pd.read_csv("../data/raw/garages.csv")
adjusters = pd.read_csv("../data/raw/adjusters.csv")

In [215]:
claims.shape, policies.shape, garages.shape, adjusters.shape

((32396, 17), (26000, 10), (70, 6), (45, 4))

checking the quality of the data

In [216]:
claims.info()

<class 'pandas.DataFrame'>
RangeIndex: 32396 entries, 0 to 32395
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   claim_id              32396 non-null  str    
 1   policy_id             32396 non-null  str    
 2   garage_id             32396 non-null  str    
 3   adjuster_id           32396 non-null  str    
 4   incident_type         32396 non-null  str    
 5   incident_date         32396 non-null  str    
 6   incident_hour         32396 non-null  int64  
 7   reported_date         32396 non-null  str    
 8   claim_amount_xaf      32396 non-null  str    
 9   police_report         32396 non-null  str    
 10  witness_count         32396 non-null  int64  
 11  prior_claims_holder   32396 non-null  int64  
 12  vehicle_towed         32396 non-null  str    
 13  investigation_opened  32396 non-null  bool   
 14  days_to_settle        31845 non-null  float64
 15  amount_paid_xaf       32396 no

In [217]:
claims.isnull().sum()

claim_id                  0
policy_id                 0
garage_id                 0
adjuster_id               0
incident_type             0
incident_date             0
incident_hour             0
reported_date             0
claim_amount_xaf          0
police_report             0
witness_count             0
prior_claims_holder       0
vehicle_towed             0
investigation_opened      0
days_to_settle          551
amount_paid_xaf           0
fraud_flag                0
dtype: int64

In [218]:
claims.duplicated().sum()

np.int64(220)

In [219]:
claims["claim_id"].duplicated().sum()

np.int64(220)

In [220]:
duplicate_rows = claims[claims.duplicated(keep=False)]

duplicate_rows.groupby("claim_id").size().value_counts().sort_index()

2    220
Name: count, dtype: int64

In [ ]:
#removing the duplicates from the claims.csv
claims = claims.drop_duplicates()

In [223]:
claims.shape

(32176, 17)

In [222]:
claims.duplicated().sum()

np.int64(0)

Checking the target

In [224]:
claims["fraud_flag"].value_counts()

fraud_flag
NO     31196
YES      980
Name: count, dtype: int64

In [225]:
claims["fraud_flag"].value_counts(normalize=True)

fraud_flag
NO     0.969543
YES    0.030457
Name: proportion, dtype: float64

In [226]:
#checking it again to see if the duplicated values are still there
claims.isnull().sum()

claim_id                  0
policy_id                 0
garage_id                 0
adjuster_id               0
incident_type             0
incident_date             0
incident_hour             0
reported_date             0
claim_amount_xaf          0
police_report             0
witness_count             0
prior_claims_holder       0
vehicle_towed             0
investigation_opened      0
days_to_settle          548
amount_paid_xaf           0
fraud_flag                0
dtype: int64

In [227]:
policies.isnull().sum()

policy_id             0
holder_id             0
region                0
vehicle_make          0
vehicle_year          0
cover_type            0
sum_insured_xaf       0
annual_premium_xaf    0
policy_start          0
payment_frequency     0
dtype: int64

In [228]:
garages.isnull().sum()

garage_id          0
garage_name        0
town               0
registered_year    0
bay_count          0
approved           0
dtype: int64

In [229]:
adjusters.isnull().sum()

adjuster_id      0
region           0
hired_year       0
caseload_band    0
dtype: int64

In [ ]:
#checking the ids before merging to ensure that they are unique
claims["claim_id"].nunique(), policies["policy_id"].nunique(), garages["garage_id"].nunique(), adjusters["adjuster_id"].nunique()

(32176, 26000, 70, 45)

In [231]:
claims["policy_id"].isin(policies["policy_id"]).value_counts()

policy_id
True     32175
False        1
Name: count, dtype: int64

In [232]:
claims["garage_id"].isin(garages["garage_id"]).value_counts()

garage_id
True    32176
Name: count, dtype: int64

In [233]:
claims["adjuster_id"].isin(adjusters["adjuster_id"]).value_counts()

adjuster_id
True    32176
Name: count, dtype: int64

In [234]:
claims["policy_id"].isin(policies["policy_id"]).sum()

np.int64(32175)

Merging the datasets

In [235]:
model_data = claims.merge(
    policies,
    on="policy_id",
    how="left"
)

model_data = model_data.merge(
    garages,
    on="garage_id",
    how="left"
)

model_data = model_data.merge(
    adjusters,
    on="adjuster_id",
    how="left"
)

In [236]:
model_data.shape

(32176, 34)

In [237]:
model_data["claim_id"].nunique()

32176

In [238]:
model_data.columns.tolist()

['claim_id',
 'policy_id',
 'garage_id',
 'adjuster_id',
 'incident_type',
 'incident_date',
 'incident_hour',
 'reported_date',
 'claim_amount_xaf',
 'police_report',
 'witness_count',
 'prior_claims_holder',
 'vehicle_towed',
 'investigation_opened',
 'days_to_settle',
 'amount_paid_xaf',
 'fraud_flag',
 'holder_id',
 'region_x',
 'vehicle_make',
 'vehicle_year',
 'cover_type',
 'sum_insured_xaf',
 'annual_premium_xaf',
 'policy_start',
 'payment_frequency',
 'garage_name',
 'town',
 'registered_year',
 'bay_count',
 'approved',
 'region_y',
 'hired_year',
 'caseload_band']

In [431]:
# Feature engineering

X = pd.DataFrame(index=model_data.index)

# Claim-level features
X["incident_type"] = model_data["incident_type"]
X["incident_hour"] = model_data["incident_hour"]
X["police_report"] = model_data["police_report"]
X["witness_count"] = model_data["witness_count"]
X["prior_claims_holder"] = model_data["prior_claims_holder"]
X["vehicle_towed"] = model_data["vehicle_towed"]

# Policy features
X["region"] = model_data["region_x"]
X["vehicle_make"] = model_data["vehicle_make"]
X["vehicle_year"] = model_data["vehicle_year"]
X["cover_type"] = model_data["cover_type"]
X["sum_insured_xaf"] = model_data["sum_insured_xaf"]
X["annual_premium_xaf"] = model_data["annual_premium_xaf"]
X["payment_frequency"] = model_data["payment_frequency"]

# Garage features
X["garage_name"] = model_data["garage_name"]
X["town"] = model_data["town"]
X["registered_year"] = model_data["registered_year"]
X["bay_count"] = model_data["bay_count"]
X["approved"] = model_data["approved"]

# Adjuster features
X["adjuster_region"] = model_data["region_y"]
X["hired_year"] = model_data["hired_year"]
X["caseload_band"] = model_data["caseload_band"]

# Claim amount
X["claim_amount_clean"] = (
    model_data["claim_amount_xaf"]
    .str.replace(",", "", regex=False)
    .str.replace("XAF", "", regex=False)
    .str.strip()
    .astype(float)
)

# Domain features
X["claim_to_insured_ratio"] = (
    X["claim_amount_clean"] / model_data["sum_insured_xaf"]
)

X["reporting_delay"] = (
    model_data["reported_date_clean"]
    - model_data["incident_date_clean"]
).dt.days

X["policy_age_days"] = (
    model_data["incident_date_clean"]
    - model_data["policy_start_clean"]
).dt.days

X["vehicle_age"] = (
    model_data["incident_date_clean"].dt.year
    - model_data["vehicle_year"]
)

# Time features
X["incident_month"] = model_data["incident_date_clean"].dt.month
X["incident_dayofweek"] = model_data["incident_date_clean"].dt.dayofweek

# Required night-hour feature
X["night_hour"] = (
    (X["incident_hour"] < 6)
    | (X["incident_hour"] >= 22)
)

# Data-quality indicators
X["late_report"] = X["reporting_delay"] > 7
X["negative_reporting_delay"] = X["reporting_delay"] < 0
X["negative_policy_age"] = X["policy_age_days"] < 0

In [432]:
# Historical entity features

history_data = model_data.sort_values(
    ["holder_id", "incident_date_clean"]
).copy()

history_data["holder_claim_count"] = (
    history_data.groupby("holder_id").cumcount()
)

history_data["holder_first_claim_date"] = (
    history_data.groupby("holder_id")["incident_date_clean"]
    .transform("min")
)

history_data["holder_history_days"] = (
    history_data["incident_date_clean"]
    - history_data["holder_first_claim_date"]
).dt.days

history_data["holder_claim_frequency"] = (
    history_data["holder_claim_count"]
    / history_data["holder_history_days"].replace(0, np.nan)
)

history_data = history_data.sort_index()

X["holder_claim_count"] = history_data["holder_claim_count"]
X["holder_history_days"] = history_data["holder_history_days"]
X["holder_claim_frequency"] = history_data["holder_claim_frequency"]

In [433]:
# Target

y = model_data["fraud_flag"].map({
    "NO": 0,
    "YES": 1
})

In [434]:
X.shape

(32176, 35)

In [435]:
X.columns.tolist()

['incident_type',
 'incident_hour',
 'police_report',
 'witness_count',
 'prior_claims_holder',
 'vehicle_towed',
 'region',
 'vehicle_make',
 'vehicle_year',
 'cover_type',
 'sum_insured_xaf',
 'annual_premium_xaf',
 'payment_frequency',
 'garage_name',
 'town',
 'registered_year',
 'bay_count',
 'approved',
 'adjuster_region',
 'hired_year',
 'caseload_band',
 'claim_amount_clean',
 'claim_to_insured_ratio',
 'reporting_delay',
 'policy_age_days',
 'vehicle_age',
 'incident_month',
 'incident_dayofweek',
 'night_hour',
 'late_report',
 'negative_reporting_delay',
 'negative_policy_age',
 'holder_claim_count',
 'holder_history_days',
 'holder_claim_frequency']

In [436]:
X.dtypes.value_counts()

str        12
float64    10
int64       7
bool        4
int32       2
Name: count, dtype: int64

In [437]:
# Leakage test

post_assessment_columns = [
    "investigation_opened",
    "days_to_settle",
    "amount_paid_xaf",
    "fraud_flag"
]

leakage_present = [
    column for column in post_assessment_columns
    if column in X.columns
]

leakage_present

[]

In [438]:
assert not leakage_present, (
    f"Post-assessment columns reached the model matrix: {leakage_present}"
)

In [439]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Preprocessing

In [440]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_columns = X_train.select_dtypes(
    include=["int64", "float64", "bool"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_columns),
    ("categorical", categorical_preprocessor, categorical_columns)
])

/var/folders/yw/1_m9x15n30b4vv_cc8hvqcg80000gn/T/ipykernel_2798/573284872.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


In [441]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [444]:
logistic_pipeline.fit(X_train, y_train)

logistic_test_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

logistic_test_pr_auc = average_precision_score(
    y_test,
    logistic_test_probabilities
)

logistic_test_roc_auc = roc_auc_score(
    y_test,
    logistic_test_probabilities
)

logistic_test_pr_auc, logistic_test_roc_auc

(0.1490229512622011, 0.8015559589220304)

In [445]:
logistic_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](35,)","['incident_type','incident_hour','police_report',...,'holder_claim_count', 'holder_history_days','holder_claim_frequency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,35
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default

In [446]:
logistic_pipeline.named_steps

{'preprocessor': ColumnTransformer(transformers=[('numeric',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median')),
                                                  ('scaler', StandardScaler())]),
                                  ['incident_hour', 'witness_count',
                                   'prior_claims_holder', 'vehicle_year',
                                   'sum_insured_xaf', 'annual_premium_xaf',
                                   'registered_year', 'bay_count', 'hired_year',
                                   'claim_amount_clean',
                                   'claim_to_insured_ratio', 'reporting_delay',...
                                   'holder_history_days',
                                   'holder_claim_frequency']),
                                 ('categorical',
                                  Pipeline(steps=[('imputer',
                                     

In [447]:
forbidden_columns = [
    "investigation_opened",
    "days_to_settle",
    "amount_paid_xaf",
    "fraud_flag"
]

forbidden_in_X = [
    col for col in forbidden_columns
    if col in X.columns
]

forbidden_in_X

[]

In [448]:
numeric_columns, categorical_columns

(['incident_hour',
  'witness_count',
  'prior_claims_holder',
  'vehicle_year',
  'sum_insured_xaf',
  'annual_premium_xaf',
  'registered_year',
  'bay_count',
  'hired_year',
  'claim_amount_clean',
  'claim_to_insured_ratio',
  'reporting_delay',
  'policy_age_days',
  'vehicle_age',
  'night_hour',
  'late_report',
  'negative_reporting_delay',
  'negative_policy_age',
  'holder_claim_count',
  'holder_history_days',
  'holder_claim_frequency'],
 ['incident_type',
  'police_report',
  'vehicle_towed',
  'region',
  'vehicle_make',
  'cover_type',
  'payment_frequency',
  'garage_name',
  'town',
  'approved',
  'adjuster_region',
  'caseload_band'])

In [449]:
final_feature_columns = X.columns.tolist()

len(final_feature_columns), final_feature_columns


(35,
 ['incident_type',
  'incident_hour',
  'police_report',
  'witness_count',
  'prior_claims_holder',
  'vehicle_towed',
  'region',
  'vehicle_make',
  'vehicle_year',
  'cover_type',
  'sum_insured_xaf',
  'annual_premium_xaf',
  'payment_frequency',
  'garage_name',
  'town',
  'registered_year',
  'bay_count',
  'approved',
  'adjuster_region',
  'hired_year',
  'caseload_band',
  'claim_amount_clean',
  'claim_to_insured_ratio',
  'reporting_delay',
  'policy_age_days',
  'vehicle_age',
  'incident_month',
  'incident_dayofweek',
  'night_hour',
  'late_report',
  'negative_reporting_delay',
  'negative_policy_age',
  'holder_claim_count',
  'holder_history_days',
  'holder_claim_frequency'])

In [450]:
required_features = [
    "claim_amount_clean",
    "claim_to_insured_ratio",
    "reporting_delay",
    "policy_age_days",
    "night_hour",
    "holder_claim_count",
    "holder_history_days",
    "holder_claim_frequency"
]

missing_required_features = [
    column for column in required_features
    if column not in final_feature_columns
]

missing_required_features

[]

In [451]:
final_feature_contract = pd.DataFrame({
    "feature": final_feature_columns,
    "type": [
        "numeric" if feature in numeric_columns else "categorical"
        for feature in final_feature_columns
    ]
})

final_feature_contract

,feature,type
0,incident_type,categorical
1,incident_hour,numeric
2,police_report,categorical
3,witness_count,numeric
4,prior_claims_holder,numeric
5,vehicle_towed,categorical
6,region,categorical
7,vehicle_make,categorical
8,vehicle_year,numeric
9,cover_type,categorical


In [452]:
test_predictions = logistic_pipeline.predict(X_test)
test_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

len(test_predictions), len(test_probabilities)

(6436, 6436)

In [453]:
average_precision_score(y_test, test_probabilities)

0.1490229512622011

In [454]:
assert "preprocessor" in logistic_pipeline.named_steps
assert "model" in logistic_pipeline.named_steps

assert not any(
    column in final_feature_columns
    for column in [
        "investigation_opened",
        "days_to_settle",
        "amount_paid_xaf",
        "fraud_flag"
    ]
)

print("Rochelle's feature pipeline checks passed.")

Rochelle's feature pipeline checks passed.
